<!-- SPDX-License-Identifier: AGPL-3.0-or-later -->
<!-- Commercial license available -->
<!-- Copyright 1998-2026 Miroslav Sotek. All rights reserved. -->

# Golden Path Evidence Notebook

This notebook is the compact SC-NeuroCore path from a deterministic trained reference model to stochastic-computing simulation, Q-format quantisation, typed IR checking, generated Verilog, and a machine-readable evidence manifest.

## Evidence Boundary

This notebook proves only what it executes locally:

- deterministic NumPy training for a small non-negative reference model;
- Q8.8 quantisation of the trained weights;
- stochastic bitstream multiplication with committed SC primitives;
- typed IR compatibility checking before hardware emission;
- equation-to-Verilog generation through the committed compiler;
- an evidence manifest with hashes and measured local errors.

It does not claim physical FPGA timing, routed area, dynamic power, IBM QPU execution, or external molecular data. Those require their own captured hardware or cloud artifacts.

In [ ]:
from __future__ import annotations

import hashlib
import json
import sys
import tempfile
from pathlib import Path

import numpy as np

REPO = Path.cwd()
if not (REPO / "src" / "sc_neurocore").exists():
    REPO = REPO.parent
if str(REPO / "src") not in sys.path:
    sys.path.insert(0, str(REPO / "src"))

from sc_neurocore.compiler.equation_compiler import Q88, equation_to_fpga
from sc_neurocore.compiler.ir_type_checker import IREdge, IRNode, SignalType, check_ir_types
from sc_neurocore.utils.bitstreams import bitstream_to_probability, generate_bernoulli_bitstream
from sc_neurocore.utils.rng import RNG

print(f"Repository root: {REPO}")

## 1. Deterministic Reference Training

The task is deliberately small and non-negative so it maps honestly into unipolar stochastic computing. The target is a two-input weighted probability estimator.

In [ ]:
rng = np.random.default_rng(314159)
X = rng.uniform(0.0, 1.0, size=(24, 2))
true_weights = np.array([0.65, 0.35], dtype=float)
target = (X @ true_weights) / len(true_weights)

weights = np.array([0.5, 0.5], dtype=float)
loss_history: list[float] = []
for _ in range(300):
    prediction = (X @ weights) / len(weights)
    error = prediction - target
    loss_history.append(float(np.mean(error * error)))
    gradient = (2.0 / len(X)) * (X.T @ error) / len(weights)
    weights = np.clip(weights - 0.8 * gradient, 0.0, 1.0)

final_prediction = (X @ weights) / len(weights)
final_mse = float(np.mean((final_prediction - target) ** 2))

assert final_mse < loss_history[0]
assert np.all((0.0 <= weights) & (weights <= 1.0))

print("trained weights:", weights)
print("initial MSE:", loss_history[0])
print("final MSE:", final_mse)

## 2. Q8.8 Quantisation

The compiler's `Q88` helper supplies the same fixed-point encoding boundary used by equation-to-Verilog generation.

In [ ]:
q88 = Q88()
weight_q88 = np.array([q88.encode(float(value)) for value in weights], dtype=np.uint16)
weight_dequantised = weight_q88.astype(float) / (1 << q88.fraction)
quantisation_error = np.abs(weight_dequantised - weights)

assert float(np.max(quantisation_error)) <= q88.resolution / 2.0 + 1e-12

print("Q8.8 weights:", weight_q88.tolist())
print("dequantised weights:", weight_dequantised.tolist())
print("max quantisation error:", float(np.max(quantisation_error)))

## 3. Stochastic-Computing Simulation

Each input and weight is encoded as an independent Bernoulli bitstream. AND implements unipolar multiplication, and popcount/mean decodes the product probability.

In [ ]:
def sc_dot(row: np.ndarray, row_index: int, weights: np.ndarray, *, length: int = 8192) -> float:
    terms: list[float] = []
    for term_index, (input_value, weight_value) in enumerate(zip(row, weights)):
        input_bits = generate_bernoulli_bitstream(
            float(input_value), length, RNG(seed=10_000 + row_index * 100 + term_index)
        )
        weight_bits = generate_bernoulli_bitstream(
            float(weight_value), length, RNG(seed=20_000 + row_index * 100 + term_index)
        )
        terms.append(bitstream_to_probability(input_bits & weight_bits))
    return float(sum(terms) / len(terms))


sample_X = X[:8]
float_outputs = (sample_X @ weight_dequantised) / len(weight_dequantised)
sc_outputs = np.array(
    [sc_dot(row, row_index, weight_dequantised) for row_index, row in enumerate(sample_X)]
)
max_abs_sc_error = float(np.max(np.abs(sc_outputs - float_outputs)))
mean_abs_sc_error = float(np.mean(np.abs(sc_outputs - float_outputs)))

assert max_abs_sc_error < 0.03

print("float outputs:", np.round(float_outputs, 6).tolist())
print("SC outputs:", np.round(sc_outputs, 6).tolist())
print("max abs SC error:", max_abs_sc_error)
print("mean abs SC error:", mean_abs_sc_error)

## 4. Typed IR and Verilog Emission

The IR checker catches signal-domain mistakes before compilation. The Verilog emission step uses the equation compiler on a minimal LIF equation so the notebook exercises an actual hardware-generation path.

In [ ]:
nodes = {
    "input": IRNode("input", "source", [], SignalType.RATE),
    "encoder": IRNode("encoder", "encoder", [SignalType.RATE], SignalType.BITSTREAM),
    "weight": IRNode("weight", "source", [], SignalType.RATE),
    "weight_encoder": IRNode(
        "weight_encoder", "encoder", [SignalType.RATE], SignalType.BITSTREAM
    ),
    "synapse": IRNode(
        "synapse",
        "and",
        [SignalType.BITSTREAM, SignalType.BITSTREAM],
        SignalType.BITSTREAM,
    ),
    "popcount": IRNode("popcount", "popcount", [SignalType.BITSTREAM], SignalType.RATE),
}
edges = [
    IREdge("input", "encoder"),
    IREdge("weight", "weight_encoder"),
    IREdge("encoder", "synapse", dst_port=0),
    IREdge("weight_encoder", "synapse", dst_port=1),
    IREdge("synapse", "popcount"),
]
type_errors = check_ir_types(nodes, edges)
assert type_errors == []

neuron, verilog = equation_to_fpga(
    "dv/dt = -(v - E_L)/tau_m + I/C",
    threshold="v > V_th",
    reset="v = V_reset",
    params={"E_L": -65.0, "tau_m": 10.0, "C": 1.0},
    init={"v": -65.0},
    module_name="golden_path_lif",
)
verilog_sha256 = hashlib.sha256(verilog.encode("utf-8")).hexdigest()

assert "module golden_path_lif" in verilog
assert len(verilog.splitlines()) > 20

print("IR type errors:", len(type_errors))
print("Verilog lines:", len(verilog.splitlines()))
print("Verilog SHA256:", verilog_sha256)

## 5. Evidence Manifest

The manifest is the handoff object: it records the deterministic inputs, measured local errors, and generated RTL hash without claiming anything beyond this run.

In [ ]:
manifest = {
    "schema_version": "sc-neurocore.golden-path-evidence.v1",
    "seed": 314159,
    "training": {
        "samples": int(len(X)),
        "initial_mse": float(loss_history[0]),
        "final_mse": final_mse,
        "weights": weights.tolist(),
    },
    "quantisation": {
        "format": "Q8.8",
        "encoded_weights": weight_q88.astype(int).tolist(),
        "dequantised_weights": weight_dequantised.tolist(),
        "max_abs_error": float(np.max(quantisation_error)),
    },
    "stochastic_simulation": {
        "bitstream_length": 8192,
        "sample_count": int(len(sample_X)),
        "max_abs_error_vs_float": max_abs_sc_error,
        "mean_abs_error_vs_float": mean_abs_sc_error,
    },
    "hardware_generation": {
        "ir_type_errors": len(type_errors),
        "verilog_lines": len(verilog.splitlines()),
        "verilog_sha256": verilog_sha256,
        "claim_boundary": "Generated RTL only; no synthesis, timing, routed area, or physical power claim.",
    },
}

with tempfile.TemporaryDirectory() as tmp_dir:
    manifest_path = Path(tmp_dir) / "golden_path_manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2) + "\n", encoding="utf-8")
    reloaded = json.loads(manifest_path.read_text(encoding="utf-8"))

assert reloaded == manifest
print(json.dumps(manifest, indent=2))